<a href="https://colab.research.google.com/github/SANGHATI23/neurofhir-qc/blob/main/15A2_NeuroFHIR_Review_WISH_Evidence_Cockpit_UI_(2).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# NeuroFHIR-Review — Notebook 15A2
## Evidence Cockpit UI — Structural Research-Product Redesign

This notebook replaces the interim “styled page” treatment with a real **research application shell** while preserving the frozen 12-case experiment.

### UI concept

**Left rail — Case Navigator**
- NeuroFHIR identity
- current case / 12
- compact 12-case progress grid
- current screen only
- no clickable case skipping
- no hidden condition/sequence disclosure

**Center — Evidence Workspace**
- the original participant application remains the authoritative interactive surface
- wider clinical-research canvas
- stronger evidence hierarchy
- MRI and longitudinal content get visual priority
- forms and decisions look like a review workstation, not a survey

**Right rail — Review Console**
- current case
- current screen
- only evidence types already visible on the screen
- study safeguards / export reminder
- no researcher key, no correctness labels, no expected disposition

**Top context bar**
- current case
- current visible stage
- case-level progress
- research-prototype status

### Experimental integrity

This notebook starts from the **pre-facelift frozen participant app**, not from 15A1. It injects a new application shell, CSS, and read-only UI telemetry only.

It does **not** change:
- cases;
- Evidence-First / AI-First assignment;
- exposure order;
- AI recommendation;
- QC / uncertainty / provenance content;
- judgments or action values;
- event names;
- timestamps;
- CSV / JSON export logic;
- participant allocation;
- researcher-only separation.

Existing application `<script>` blocks and all non-index assets must remain byte-for-byte unchanged.

## Order

`13 → 14 → 14B → 14C → 15 → 15A2 → 15A redeploy → 15B live QA → Step 9 reviewers → 16 analysis`

Once participant **P003** begins, freeze the interface and do not redesign it during data collection.

In [1]:
# Cell 1 — Mount Drive
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
# Cell 2 — Paths and safety configuration
from pathlib import Path
import shutil, hashlib, json, re, os, datetime

DRIVE_REPO_ROOT = Path("/content/drive/MyDrive/neurofhir-qc")
WISH_ROOT = DRIVE_REPO_ROOT / "wish_extension"

FROZEN_APP = WISH_ROOT / "final_wish_pilot" / "participant_app"
CURRENT_INDEX = FROZEN_APP / "index.html"

# Created by 15A1. If absent, this notebook can create it ONLY from a truly
# pre-facelift app.
PRE_UI_BACKUP = (
    WISH_ROOT / "final_wish_pilot" / "participant_app_pre_ui_facelift"
)

CANDIDATE_APP = (
    WISH_ROOT / "final_wish_pilot" / "participant_app_evidence_cockpit_candidate"
)

PROMOTE_TO_FROZEN_APP = True

assert CURRENT_INDEX.exists(), f"Participant app not found: {CURRENT_INDEX}"

print("Current participant app:", FROZEN_APP)
print("Pre-UI backup:", PRE_UI_BACKUP)
print("Candidate:", CANDIDATE_APP)
print("Promote after all gates:", PROMOTE_TO_FROZEN_APP)

Current participant app: /content/drive/MyDrive/neurofhir-qc/wish_extension/final_wish_pilot/participant_app
Pre-UI backup: /content/drive/MyDrive/neurofhir-qc/wish_extension/final_wish_pilot/participant_app_pre_ui_facelift
Candidate: /content/drive/MyDrive/neurofhir-qc/wish_extension/final_wish_pilot/participant_app_evidence_cockpit_candidate
Promote after all gates: True


In [3]:
# Cell 3 — Resolve the clean pre-facelift source
def sha256_file(p: Path) -> str:
    h = hashlib.sha256()
    with p.open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

current_html = CURRENT_INDEX.read_text(encoding="utf-8")

UI_MARKERS = [
    "NF_UI_FACELIFT_STYLE_BEGIN",
    "NF_UI_FACELIFT_HEADER_BEGIN",
    "NF_UI_FACELIFT_SCRIPT_BEGIN",
    "NF_UI_COCKPIT_STYLE_BEGIN",
]

if PRE_UI_BACKUP.exists():
    SOURCE_APP = PRE_UI_BACKUP
    print("✅ Using existing pre-UI backup.")
else:
    # Never make a "clean backup" from an already-modified UI.
    assert not any(m in current_html for m in UI_MARKERS), (
        "No pre-UI backup exists, but the current participant app already contains "
        "a UI facelift marker. STOP rather than stacking redesigns."
    )
    shutil.copytree(FROZEN_APP, PRE_UI_BACKUP)
    SOURCE_APP = PRE_UI_BACKUP
    print("✅ Created clean pre-UI backup:", PRE_UI_BACKUP)

SOURCE_INDEX = SOURCE_APP / "index.html"
assert SOURCE_INDEX.exists()

source_html = SOURCE_INDEX.read_text(encoding="utf-8")
source_hash = sha256_file(SOURCE_INDEX)

print("Clean source index SHA-256:", source_hash)
print("Clean source bytes:", len(source_html.encode("utf-8")))

✅ Using existing pre-UI backup.
Clean source index SHA-256: 40ed5f33e47ddea1716f4c63626c158156fdd613d177a489bbfa825e72264e29
Clean source bytes: 15181


In [4]:
# Cell 4 — Snapshot original scientific / interactive structure
from bs4 import BeautifulSoup

SCRIPT_RE = re.compile(r"<script\b[^>]*>.*?</script>", re.I | re.S)
scripts_before = SCRIPT_RE.findall(source_html)

def sha256_text(s: str) -> str:
    return hashlib.sha256(s.encode("utf-8")).hexdigest()

script_hashes_before = [sha256_text(s) for s in scripts_before]

soup_before = BeautifulSoup(source_html, "html.parser")

def control_signature(soup):
    result = []
    for el in soup.find_all(["input", "select", "textarea", "button"]):
        result.append({
            "tag": el.name,
            "id": el.get("id"),
            "name": el.get("name"),
            "type": el.get("type"),
            "value": el.get("value"),
            "text": " ".join(el.stripped_strings),
        })
    return result

controls_before = control_signature(soup_before)

critical_tokens = [
    "initial_judgment",
    "initial_confidence",
    "provenance_opened",
    "final_action",
    "final_confidence",
    "reason_code",
    "rationale",
    "initial_submitted_utc",
    "final_submitted_utc",
]

tokens_present_before = {
    t: (t.lower() in source_html.lower())
    for t in critical_tokens
}

assert len(scripts_before) >= 1
assert "neurofhir-review" in source_html.lower()

print("Existing script blocks:", len(scripts_before))
print("Existing controls:", len(controls_before))
print("Critical study tokens present:", [k for k, v in tokens_present_before.items() if v])
print("✅ Baseline snapshot: PASS")

Existing script blocks: 1
Existing controls: 0
Critical study tokens present: ['initial_judgment', 'initial_confidence', 'provenance_opened', 'final_action', 'final_confidence', 'reason_code', 'rationale', 'initial_submitted_utc', 'final_submitted_utc']
✅ Baseline snapshot: PASS


In [5]:
# Cell 5 — Evidence Cockpit visual system

COCKPIT_STYLE = r"""
<!-- NF_UI_COCKPIT_STYLE_BEGIN -->
<style id="nf-evidence-cockpit-style">
:root {
  --nf-left: 236px;
  --nf-right: 292px;
  --nf-top: 72px;

  --ink-950: #0b1721;
  --ink-900: #122330;
  --ink-800: #203645;
  --ink-700: #3f5361;
  --ink-600: #5a6d79;
  --ink-500: #748592;

  --navy-950: #071923;
  --navy-900: #0a2433;
  --navy-850: #0d2d3e;
  --teal-700: #0b6e78;
  --teal-600: #12818b;
  --teal-500: #2c98a0;
  --teal-100: #e5f4f4;
  --blue-100: #eaf2f8;

  --line-300: #c7d2d9;
  --line-200: #dbe3e8;
  --line-100: #eaf0f3;
  --surface: #ffffff;
  --surface-2: #f6f8fa;
  --surface-3: #eef3f6;

  --shadow-1: 0 1px 2px rgba(10,36,51,.05), 0 2px 8px rgba(10,36,51,.04);
  --shadow-2: 0 12px 35px rgba(10,36,51,.08), 0 2px 10px rgba(10,36,51,.05);

  --radius-sm: 9px;
  --radius-md: 13px;
  --radius-lg: 18px;
}

* { box-sizing: border-box; }

html {
  min-height: 100%;
  background: #f1f5f7;
}

body {
  min-height: 100vh;
  margin: 0 !important;
  padding:
    calc(var(--nf-top) + 26px)
    calc(var(--nf-right) + 30px)
    46px
    calc(var(--nf-left) + 30px) !important;
  color: var(--ink-950);
  background:
    radial-gradient(circle at 30% -5%, rgba(18,129,139,.055), transparent 28rem),
    #f3f6f8 !important;
  font-family:
    Inter, ui-sans-serif, -apple-system, BlinkMacSystemFont, "Segoe UI",
    Roboto, Helvetica, Arial, sans-serif !important;
  font-size: 15.5px;
  line-height: 1.55;
  -webkit-font-smoothing: antialiased;
  text-rendering: optimizeLegibility;
}

/* ================= LEFT CASE RAIL ================= */
#nf-cockpit-left {
  position: fixed;
  inset: 0 auto 0 0;
  z-index: 10000;
  width: var(--nf-left);
  padding: 20px 16px 18px;
  color: #fff;
  background:
    linear-gradient(180deg, #071923 0%, #0a2635 58%, #0c2d3c 100%);
  border-right: 1px solid rgba(255,255,255,.07);
  box-shadow: 8px 0 26px rgba(7,25,35,.08);
  overflow-y: auto;
}

.nf-brand-row {
  display: flex;
  align-items: center;
  gap: 11px;
  padding: 1px 2px 18px;
}

.nf-brand-mark {
  width: 38px;
  height: 38px;
  flex: 0 0 38px;
  display: grid;
  place-items: center;
  border: 1px solid rgba(255,255,255,.16);
  border-radius: 11px;
  background: rgba(255,255,255,.07);
  color: #ddfbfb;
  font-size: 13px;
  font-weight: 820;
  letter-spacing: -.025em;
}

.nf-brand-name {
  color: #fff;
  font-size: 14px;
  font-weight: 760;
  line-height: 1.1;
  letter-spacing: -.01em;
}

.nf-brand-mode {
  margin-top: 4px;
  color: rgba(255,255,255,.53);
  font-size: 10px;
  font-weight: 650;
  letter-spacing: .07em;
  text-transform: uppercase;
}

.nf-rail-section {
  padding: 16px 2px;
  border-top: 1px solid rgba(255,255,255,.09);
}

.nf-rail-label {
  margin-bottom: 10px;
  color: rgba(255,255,255,.45);
  font-size: 10px;
  font-weight: 760;
  letter-spacing: .09em;
  text-transform: uppercase;
}

.nf-case-large {
  display: flex;
  align-items: baseline;
  gap: 5px;
  margin-bottom: 13px;
}

#nf-case-current {
  color: #fff;
  font-size: 32px;
  font-weight: 760;
  line-height: 1;
  font-variant-numeric: tabular-nums;
}

.nf-case-total {
  color: rgba(255,255,255,.43);
  font-size: 12px;
  font-weight: 650;
}

.nf-case-grid {
  display: grid;
  grid-template-columns: repeat(4, 1fr);
  gap: 7px;
}

.nf-case-dot {
  height: 29px;
  display: grid;
  place-items: center;
  border-radius: 8px;
  color: rgba(255,255,255,.42);
  background: rgba(255,255,255,.045);
  border: 1px solid rgba(255,255,255,.07);
  font-size: 10px;
  font-weight: 720;
  font-variant-numeric: tabular-nums;
}

.nf-case-dot.is-past {
  color: #bfe6e7;
  background: rgba(44,152,160,.10);
  border-color: rgba(90,190,197,.18);
}

.nf-case-dot.is-current {
  color: #071923;
  background: #9ae1e4;
  border-color: #9ae1e4;
  box-shadow: 0 4px 14px rgba(80,190,196,.20);
}

.nf-stage-now {
  position: relative;
  padding: 11px 11px 11px 14px;
  border: 1px solid rgba(255,255,255,.10);
  border-radius: 11px;
  background: rgba(255,255,255,.055);
}

.nf-stage-now::before {
  content: "";
  position: absolute;
  left: -1px;
  top: 9px;
  bottom: 9px;
  width: 3px;
  border-radius: 3px;
  background: #7ed8dc;
}

#nf-left-stage {
  color: #fff;
  font-size: 12px;
  font-weight: 700;
}

.nf-stage-sub {
  margin-top: 4px;
  color: rgba(255,255,255,.48);
  font-size: 10.5px;
  line-height: 1.4;
}

.nf-rail-foot {
  margin-top: 4px;
  color: rgba(255,255,255,.35);
  font-size: 10px;
  line-height: 1.5;
}

/* ================= TOP CONTEXT BAR ================= */
#nf-cockpit-top {
  position: fixed;
  top: 0;
  left: var(--nf-left);
  right: var(--nf-right);
  z-index: 9999;
  height: var(--nf-top);
  display: flex;
  align-items: center;
  gap: 20px;
  padding: 0 28px;
  background: rgba(255,255,255,.94);
  border-bottom: 1px solid var(--line-200);
  box-shadow: 0 3px 14px rgba(10,36,51,.035);
  backdrop-filter: blur(12px);
}

.nf-top-context {
  min-width: 0;
  flex: 1;
}

.nf-top-eyebrow {
  color: var(--teal-700);
  font-size: 10px;
  font-weight: 780;
  letter-spacing: .08em;
  text-transform: uppercase;
}

#nf-top-stage {
  margin-top: 3px;
  color: var(--ink-900);
  font-size: 17px;
  font-weight: 740;
  letter-spacing: -.015em;
  white-space: nowrap;
  overflow: hidden;
  text-overflow: ellipsis;
}

.nf-top-progress {
  min-width: 210px;
  max-width: 280px;
  flex: 0 1 280px;
}

.nf-progress-meta {
  display: flex;
  justify-content: space-between;
  gap: 10px;
  margin-bottom: 6px;
  color: var(--ink-500);
  font-size: 10.5px;
}

.nf-progress-track {
  height: 5px;
  overflow: hidden;
  border-radius: 999px;
  background: var(--line-100);
}

#nf-progress-fill {
  width: 0%;
  height: 100%;
  border-radius: inherit;
  background: linear-gradient(90deg, var(--teal-700), #48aeb4);
  transition: width .22s ease;
}

.nf-prototype-pill {
  flex: 0 0 auto;
  padding: 7px 9px;
  border: 1px solid var(--line-200);
  border-radius: 999px;
  color: var(--ink-600);
  background: var(--surface-2);
  font-size: 10px;
  font-weight: 680;
}

/* ================= RIGHT REVIEW CONSOLE ================= */
#nf-cockpit-right {
  position: fixed;
  inset: 0 0 0 auto;
  z-index: 9998;
  width: var(--nf-right);
  padding: calc(var(--nf-top) + 18px) 18px 20px;
  color: var(--ink-900);
  background: #fafcfd;
  border-left: 1px solid var(--line-200);
  overflow-y: auto;
}

.nf-console-title {
  margin-bottom: 3px;
  color: var(--ink-900);
  font-size: 13px;
  font-weight: 770;
  letter-spacing: -.01em;
}

.nf-console-subtitle {
  margin-bottom: 18px;
  color: var(--ink-500);
  font-size: 10.5px;
}

.nf-console-card {
  margin-bottom: 12px;
  padding: 14px;
  border: 1px solid var(--line-200);
  border-radius: 13px;
  background: #fff;
  box-shadow: var(--shadow-1);
}

.nf-console-card-title {
  margin-bottom: 10px;
  color: var(--ink-500);
  font-size: 9.5px;
  font-weight: 780;
  letter-spacing: .08em;
  text-transform: uppercase;
}

.nf-kv {
  display: grid;
  grid-template-columns: 78px minmax(0,1fr);
  gap: 8px;
  padding: 7px 0;
  border-top: 1px solid var(--line-100);
  font-size: 10.5px;
}

.nf-kv:first-of-type {
  padding-top: 0;
  border-top: 0;
}

.nf-k {
  color: var(--ink-500);
  font-weight: 620;
}

.nf-v {
  color: var(--ink-900);
  font-weight: 680;
  text-align: right;
  word-break: break-word;
}

#nf-visible-evidence {
  display: flex;
  flex-wrap: wrap;
  gap: 6px;
}

.nf-evidence-chip {
  padding: 5px 7px;
  border: 1px solid #d3e2e7;
  border-radius: 7px;
  color: #34515e;
  background: #f1f7f8;
  font-size: 9.5px;
  font-weight: 690;
}

.nf-console-rule {
  color: var(--ink-600);
  font-size: 10.5px;
  line-height: 1.55;
}

.nf-safeguard {
  display: flex;
  align-items: flex-start;
  gap: 7px;
  margin-top: 8px;
  color: var(--ink-600);
  font-size: 9.8px;
  line-height: 1.45;
}

.nf-safeguard:first-of-type { margin-top: 0; }

.nf-safeguard-dot {
  width: 6px;
  height: 6px;
  flex: 0 0 6px;
  margin-top: 4px;
  border-radius: 50%;
  background: var(--teal-500);
}

/* ================= CENTER EVIDENCE WORKSPACE ================= */
body > :not([data-nf-cockpit]) {
  max-width: 1180px;
  margin-left: auto !important;
  margin-right: auto !important;
}

body > div:not([data-nf-cockpit]),
body > main:not([data-nf-cockpit]) {
  width: 100% !important;
}

h1, h2, h3, h4 {
  color: var(--ink-900);
  line-height: 1.22;
  letter-spacing: -.022em;
}

h1 {
  margin-top: 0;
  font-size: clamp(25px,2.7vw,33px);
  font-weight: 770;
}

h2 {
  font-size: clamp(20px,2.1vw,25px);
  font-weight: 740;
}

h3 {
  font-size: 16px;
  font-weight: 720;
}

p { color: var(--ink-700); }

.nf-auto-panel {
  margin: 15px 0 !important;
  padding: 21px 22px !important;
  border: 1px solid var(--line-200) !important;
  border-radius: var(--radius-lg) !important;
  background: rgba(255,255,255,.97) !important;
  box-shadow: var(--shadow-1) !important;
}

.nf-auto-panel > :first-child { margin-top: 0 !important; }
.nf-auto-panel > :last-child { margin-bottom: 0 !important; }

.nf-panel-evidence {
  border-top: 3px solid var(--teal-600) !important;
}

.nf-panel-ai {
  /* Deliberately restrained: AI is not visually privileged. */
  border-top: 3px solid #7a94a5 !important;
}

.nf-panel-provenance {
  border-top: 3px solid #607f91 !important;
}

.nf-panel-decision {
  border-top: 3px solid #314f61 !important;
}

.nf-panel-complete {
  border-top: 3px solid #3b8e79 !important;
}

.nf-panel-label {
  display: inline-flex;
  align-items: center;
  margin-bottom: 10px;
  padding: 4px 7px;
  border: 1px solid #dbe5ea;
  border-radius: 6px;
  color: var(--ink-600);
  background: #f7fafb;
  font-size: 9px;
  font-weight: 780;
  letter-spacing: .08em;
  text-transform: uppercase;
}

/* Imaging should feel like the evidence workspace, not a thumbnail page. */
.nf-auto-panel img,
.nf-auto-panel canvas {
  max-width: 100%;
  border-radius: 12px;
  box-shadow:
    0 0 0 1px rgba(10,36,51,.10),
    0 6px 18px rgba(10,36,51,.07);
}

img, canvas, svg { max-width: 100%; }

/* Data */
table {
  width: 100%;
  overflow: hidden;
  border-collapse: separate;
  border-spacing: 0;
  border: 1px solid var(--line-200);
  border-radius: 12px;
  background: #fff;
}

th, td {
  padding: 10px 12px;
  border-bottom: 1px solid var(--line-100);
  text-align: left;
  vertical-align: top;
}

th {
  color: var(--ink-600);
  background: #f7f9fa;
  font-size: 10.5px;
  font-weight: 730;
}

tr:last-child td { border-bottom: 0; }

pre {
  overflow: auto;
  padding: 14px;
  border: 1px solid var(--line-200);
  border-radius: 11px;
  color: #27404f;
  background: #f5f8fa;
  font-size: 11.5px;
  line-height: 1.5;
}

/* Forms */
label {
  color: var(--ink-900);
  font-weight: 630;
}

input[type="text"],
input[type="number"],
select,
textarea {
  width: 100%;
  padding: 10px 11px;
  border: 1px solid #bccbd4;
  border-radius: 9px;
  color: var(--ink-900);
  background: #fff;
  font: inherit;
  outline: 0;
  transition: border-color .15s ease, box-shadow .15s ease;
}

textarea {
  min-height: 108px;
  resize: vertical;
}

input[type="text"]:focus,
input[type="number"]:focus,
select:focus,
textarea:focus {
  border-color: var(--teal-600);
  box-shadow: 0 0 0 3px rgba(18,129,139,.12);
}

input[type="radio"],
input[type="checkbox"] {
  width: 17px;
  height: 17px;
  accent-color: var(--teal-700);
}

fieldset {
  margin: 15px 0;
  padding: 16px;
  border: 1px solid var(--line-200);
  border-radius: 12px;
  background: #fbfcfd;
}

legend {
  padding: 0 5px;
  color: var(--ink-900);
  font-weight: 700;
}

/* Buttons */
button,
input[type="button"],
input[type="submit"] {
  min-height: 41px;
  padding: 9px 15px;
  border: 1px solid transparent;
  border-radius: 9px;
  color: #fff;
  background: #163e50;
  font: inherit;
  font-weight: 680;
  cursor: pointer;
  box-shadow: 0 1px 2px rgba(10,36,51,.08);
  transition: background .15s ease, box-shadow .15s ease, transform .08s ease;
}

button:hover,
input[type="button"]:hover,
input[type="submit"]:hover {
  background: #0e3142;
  box-shadow: 0 5px 14px rgba(10,36,51,.12);
}

button:active,
input[type="button"]:active,
input[type="submit"]:active {
  transform: translateY(1px);
}

button:disabled,
input[type="submit"]:disabled {
  opacity: .48;
  cursor: not-allowed;
  box-shadow: none;
}

.nf-secondary-action {
  color: #234555 !important;
  background: #fff !important;
  border-color: #bfcdd5 !important;
}

.nf-secondary-action:hover {
  background: #f5f8fa !important;
}

button:focus-visible,
input:focus-visible,
select:focus-visible,
textarea:focus-visible,
a:focus-visible {
  outline: 3px solid rgba(18,129,139,.24);
  outline-offset: 2px;
}

/* ================= RESPONSIVE ================= */
@media (max-width: 1220px) {
  :root { --nf-right: 0px; }
  #nf-cockpit-right { display: none; }
}

@media (max-width: 860px) {
  :root {
    --nf-left: 0px;
    --nf-top: 64px;
  }

  body {
    padding:
      calc(var(--nf-top) + 76px)
      14px
      30px
      14px !important;
  }

  #nf-cockpit-left {
    inset: var(--nf-top) 0 auto 0;
    width: auto;
    height: 64px;
    display: flex;
    align-items: center;
    gap: 13px;
    padding: 10px 14px;
    overflow: hidden;
  }

  .nf-brand-row {
    flex: 0 0 auto;
    padding: 0 12px 0 0;
    border-right: 1px solid rgba(255,255,255,.10);
  }

  .nf-brand-mark {
    width: 32px;
    height: 32px;
    flex-basis: 32px;
  }

  .nf-brand-mode,
  .nf-rail-section:not(.nf-mobile-current),
  .nf-rail-foot {
    display: none;
  }

  .nf-mobile-current {
    display: flex;
    align-items: center;
    gap: 10px;
    padding: 0;
    border: 0;
  }

  .nf-case-large { margin: 0; }
  #nf-case-current { font-size: 20px; }
  .nf-stage-now { padding: 7px 9px 7px 12px; }

  #nf-cockpit-top {
    left: 0;
    right: 0;
    padding: 0 14px;
  }

  .nf-top-progress { min-width: 125px; flex-basis: 170px; }
  .nf-prototype-pill { display: none; }
}

@media (max-width: 560px) {
  .nf-top-progress { display: none; }
  #nf-top-stage { font-size: 14px; }
  .nf-brand-name { font-size: 12px; }
  .nf-stage-now { display: none; }

  .nf-auto-panel {
    padding: 16px 14px !important;
    border-radius: 14px !important;
  }

  button,
  input[type="button"],
  input[type="submit"] {
    width: 100%;
  }
}

@media (prefers-reduced-motion: reduce) {
  *,*::before,*::after {
    animation: none !important;
    transition: none !important;
    scroll-behavior: auto !important;
  }
}

@media print {
  #nf-cockpit-left,
  #nf-cockpit-top,
  #nf-cockpit-right {
    display: none !important;
  }
  body {
    padding: 0 !important;
    background: #fff !important;
  }
  .nf-auto-panel { box-shadow: none !important; }
}
</style>
<!-- NF_UI_COCKPIT_STYLE_END -->
"""

print("✅ Evidence Cockpit stylesheet defined.")

✅ Evidence Cockpit stylesheet defined.


In [6]:
# Cell 6 — Read-only application shell
#
# The shell contains no form controls and no navigation links.
# It cannot submit, skip, alter, or reveal a case.

LEFT_RAIL = r"""
<!-- NF_UI_COCKPIT_LEFT_BEGIN -->
<aside id="nf-cockpit-left" data-nf-cockpit="true" aria-label="Study session">
  <div class="nf-brand-row">
    <div class="nf-brand-mark" aria-hidden="true">NF</div>
    <div>
      <div class="nf-brand-name">NeuroFHIR-Review</div>
      <div class="nf-brand-mode">Evidence cockpit</div>
    </div>
  </div>

  <section class="nf-rail-section nf-mobile-current">
    <div>
      <div class="nf-rail-label">Current case</div>
      <div class="nf-case-large">
        <span id="nf-case-current">—</span>
        <span class="nf-case-total">/ 12</span>
      </div>
    </div>
    <div class="nf-stage-now">
      <div id="nf-left-stage">Case review</div>
      <div class="nf-stage-sub">Only the current visible stage is shown.</div>
    </div>
  </section>

  <section class="nf-rail-section">
    <div class="nf-rail-label">Case navigator</div>
    <div id="nf-case-grid" class="nf-case-grid" aria-label="12-case progress">
      <span class="nf-case-dot" data-case="1">01</span>
      <span class="nf-case-dot" data-case="2">02</span>
      <span class="nf-case-dot" data-case="3">03</span>
      <span class="nf-case-dot" data-case="4">04</span>
      <span class="nf-case-dot" data-case="5">05</span>
      <span class="nf-case-dot" data-case="6">06</span>
      <span class="nf-case-dot" data-case="7">07</span>
      <span class="nf-case-dot" data-case="8">08</span>
      <span class="nf-case-dot" data-case="9">09</span>
      <span class="nf-case-dot" data-case="10">10</span>
      <span class="nf-case-dot" data-case="11">11</span>
      <span class="nf-case-dot" data-case="12">12</span>
    </div>
  </section>

  <section class="nf-rail-section">
    <div class="nf-rail-label">Current screen</div>
    <div class="nf-stage-now">
      <div id="nf-left-stage-2">Case review</div>
      <div class="nf-stage-sub">Use only the information available on this screen.</div>
    </div>
  </section>

  <div class="nf-rail-foot">
    Research prototype<br>
    Not for patient care or diagnostic use
  </div>
</aside>
<!-- NF_UI_COCKPIT_LEFT_END -->
"""

TOP_BAR = r"""
<!-- NF_UI_COCKPIT_TOP_BEGIN -->
<header id="nf-cockpit-top" data-nf-cockpit="true" aria-label="Current review context">
  <div class="nf-top-context">
    <div class="nf-top-eyebrow">Human–AI evidence review</div>
    <div id="nf-top-stage">Case review</div>
  </div>

  <div class="nf-top-progress" aria-label="Case progress">
    <div class="nf-progress-meta">
      <span id="nf-progress-case">Case — of 12</span>
      <span id="nf-progress-percent">0%</span>
    </div>
    <div class="nf-progress-track" aria-hidden="true">
      <div id="nf-progress-fill"></div>
    </div>
  </div>

  <div class="nf-prototype-pill">Research prototype</div>
</header>
<!-- NF_UI_COCKPIT_TOP_END -->
"""

RIGHT_RAIL = r"""
<!-- NF_UI_COCKPIT_RIGHT_BEGIN -->
<aside id="nf-cockpit-right" data-nf-cockpit="true" aria-label="Review console">
  <div class="nf-console-title">Review console</div>
  <div class="nf-console-subtitle">Read-only session context</div>

  <section class="nf-console-card">
    <div class="nf-console-card-title">Session</div>
    <div class="nf-kv">
      <div class="nf-k">Case</div>
      <div id="nf-console-case" class="nf-v">— / 12</div>
    </div>
    <div class="nf-kv">
      <div class="nf-k">Screen</div>
      <div id="nf-console-stage" class="nf-v">Case review</div>
    </div>
    <div class="nf-kv">
      <div class="nf-k">Status</div>
      <div id="nf-console-status" class="nf-v">In progress</div>
    </div>
  </section>

  <section class="nf-console-card">
    <div class="nf-console-card-title">Visible evidence</div>
    <div id="nf-visible-evidence">
      <span class="nf-evidence-chip">Current screen</span>
    </div>
  </section>

  <section class="nf-console-card">
    <div class="nf-console-card-title">Review rule</div>
    <div class="nf-console-rule">
      Base each judgment only on information visible at the current stage.
      The interface does not reveal the hidden study sequence.
    </div>
  </section>

  <section class="nf-console-card">
    <div class="nf-console-card-title">Study safeguards</div>
    <div class="nf-safeguard">
      <span class="nf-safeguard-dot"></span>
      <span>Researcher answer key is not included in this application.</span>
    </div>
    <div class="nf-safeguard">
      <span class="nf-safeguard-dot"></span>
      <span>Case order and evidence exposure remain controlled by the frozen study logic.</span>
    </div>
    <div class="nf-safeguard">
      <span class="nf-safeguard-dot"></span>
      <span>Responses are exported at session completion using the study controls.</span>
    </div>
  </section>
</aside>
<!-- NF_UI_COCKPIT_RIGHT_END -->
"""

print("✅ Left rail, top context bar, and right review console defined.")

✅ Left rail, top context bar, and right review console defined.


In [7]:
# Cell 7 — Runtime layout intelligence (CORRECTED: no future-stage leakage)
#
# IMPORTANT:
# The study's step navigator can visibly contain labels for FUTURE stages
# ("AI review", "Evidence Passport", "Final action"). Those labels are navigation
# chrome, NOT evidence currently available to the participant.
#
# This UI layer therefore determines the current stage from:
#   1) the visible CURRENT CONTENT HEADING;
#   2) an explicitly active/current step element if present;
#   3) a conservative fallback.
#
# The right-rail evidence chips are generated from the CURRENT STAGE, not from
# every visible word on the page. This prevents future-stage priming/leakage.
#
# This script never changes participant inputs or the frozen study sequence.

COCKPIT_SCRIPT = r"""
<!-- NF_UI_COCKPIT_SCRIPT_BEGIN -->
<script id="nf-evidence-cockpit-script">
(function () {
  "use strict";

  function isCockpit(el) {
    return !!(el && el.closest && el.closest("[data-nf-cockpit]"));
  }

  function isVisible(el) {
    if (!el || isCockpit(el)) return false;

    var style = window.getComputedStyle(el);
    if (
      style.display === "none" ||
      style.visibility === "hidden" ||
      style.opacity === "0"
    ) return false;

    var r = el.getBoundingClientRect();
    return (r.width > 0 && r.height > 0);
  }

  function cleanText(el) {
    return (el && el.textContent ? el.textContent : "")
      .replace(/\s+/g, " ")
      .trim();
  }

  function visibleStudyText() {
    var parts = [];

    document.querySelectorAll(
      "h1,h2,h3,h4,h5,p,label,legend,td,th,li"
    ).forEach(function (el) {
      if (!isVisible(el)) return;
      var t = cleanText(el);
      if (t) parts.push(t);
    });

    return parts.join(" | ");
  }

  function visibleHeadingTexts() {
    var headings = [];

    document.querySelectorAll("h1,h2,h3,h4,h5,legend,[role='heading']").forEach(function (el) {
      if (!isVisible(el)) return;
      var t = cleanText(el);
      if (t) headings.push(t);
    });

    return headings;
  }

  function findCurrentCase(text) {
    var patterns = [
      /review\s+case\s+0?(\d+)/i,
      /case\s+0?(\d+)\s+of\s+12/i,
      /case\s+0?(\d+)\s*\/\s*12/i,
      /case\s*#?\s*0?(\d+)/i
    ];

    for (var i = 0; i < patterns.length; i++) {
      var m = text.match(patterns[i]);
      if (m) {
        var n = parseInt(m[1], 10);
        if (n >= 1 && n <= 12) return n;
      }
    }

    return null;
  }

  function stageFromText(raw) {
    var t = (raw || "").toLowerCase();

    if (/study setup|participant id|reviewer category/i.test(t)) return "Study setup";

    if (
      /review complete|session complete|completion/i.test(t)
    ) return "Review complete";

    if (
      /final disposition|final action|final decision/i.test(t)
    ) return "Final disposition";

    if (
      /evidence passport|provenance inspection|evidence provenance/i.test(t)
    ) return "Evidence provenance";

    if (
      /ai recommendation|ai review|ai evidence review/i.test(t)
    ) return "AI evidence review";

    if (
      /initial judgment|first judgment/i.test(t)
    ) return "Initial judgment";

    if (
      /evidence review|longitudinal evidence|review evidence/i.test(t)
    ) return "Evidence review";

    if (
      /case brief|case overview/i.test(t)
    ) return "Case brief";

    return null;
  }

  function detectStage() {
    // 1) Prefer visible CONTENT HEADINGS.
    // Scan from deepest/smaller headings upward because the page H1 is often
    // the product title while H2/H3 contains the active study screen.
    var headings = visibleHeadingTexts();

    for (var i = headings.length - 1; i >= 0; i--) {
      var stage = stageFromText(headings[i]);
      if (stage) return stage;
    }

    // 2) If the application marks a current step explicitly, use it.
    var activeSelectors = [
      '[aria-current="step"]',
      '[aria-selected="true"]',
      '.active',
      '.current',
      '.selected'
    ];

    for (var s = 0; s < activeSelectors.length; s++) {
      var activeEls = document.querySelectorAll(activeSelectors[s]);
      for (var j = 0; j < activeEls.length; j++) {
        var el = activeEls[j];
        if (!isVisible(el)) continue;
        var activeStage = stageFromText(cleanText(el));
        if (activeStage) return activeStage;
      }
    }

    // 3) Conservative fallback: do NOT infer a later stage from the global
    // step navigator. "Case review" leaks nothing about future evidence.
    return "Case review";
  }

  function evidenceForStage(stage) {
    // IMPORTANT:
    // Only describe evidence that is legitimately available by THIS stage.
    // Do not parse the global step navigator for evidence labels.
    switch (stage) {
      case "Study setup":
        return ["No case evidence yet"];

      case "Case brief":
        return ["Case context"];

      case "Evidence review":
        return ["Case evidence", "Longitudinal evidence"];

      case "Initial judgment":
        return ["Case evidence", "Evidence reviewed"];

      case "AI evidence review":
        return ["Case evidence", "AI recommendation"];

      case "Evidence provenance":
        return ["Case evidence", "AI recommendation", "Provenance"];

      case "Final disposition":
        return ["Case evidence", "AI recommendation", "Provenance"];

      case "Review complete":
        return ["Session complete"];

      default:
        return ["Current screen"];
    }
  }


  function normalizeSetupPresentation(stage) {
    if (stage !== "Study setup") return;

    // P001/P002 are reserved QA IDs. Do not suggest them to real reviewers.
    // This changes placeholder text only; it never sets a participant value.
    var candidates = document.querySelectorAll(
      'input[id*="participant" i], input[name*="participant" i], input[placeholder="P001"]'
    );

    candidates.forEach(function (input) {
      if (input.value) return;
      var ph = (input.getAttribute("placeholder") || "").trim();
      if (!ph || /^P00[12]$/i.test(ph)) {
        input.setAttribute("placeholder", "Assigned ID (P003+)");
      }
    });
  }

  function updateCaseUI(caseNo) {
    var display = caseNo ? String(caseNo).padStart(2, "0") : "—";

    var current = document.getElementById("nf-case-current");
    var consoleCase = document.getElementById("nf-console-case");
    var progressCase = document.getElementById("nf-progress-case");
    var progressPct = document.getElementById("nf-progress-percent");
    var fill = document.getElementById("nf-progress-fill");

    if (current) current.textContent = display;
    if (consoleCase) consoleCase.textContent = (caseNo ? caseNo : "—") + " / 12";
    if (progressCase) progressCase.textContent = "Case " + (caseNo ? caseNo : "—") + " of 12";

    var pct = caseNo ? Math.round((caseNo / 12) * 100) : 0;
    if (progressPct) progressPct.textContent = pct + "%";
    if (fill) fill.style.width = pct + "%";

    document.querySelectorAll(".nf-case-dot").forEach(function (dot) {
      var n = parseInt(dot.getAttribute("data-case"), 10);
      dot.classList.remove("is-past", "is-current");

      if (caseNo && n < caseNo) dot.classList.add("is-past");
      if (caseNo && n === caseNo) dot.classList.add("is-current");
    });
  }

  function updateStageUI(stage) {
    [
      "nf-left-stage",
      "nf-left-stage-2",
      "nf-top-stage",
      "nf-console-stage"
    ].forEach(function (id) {
      var el = document.getElementById(id);
      if (el) el.textContent = stage;
    });

    var status = document.getElementById("nf-console-status");
    if (status) {
      status.textContent =
        stage === "Review complete" ? "Complete" :
        stage === "Study setup" ? "Not started" : "In progress";
    }

    var visibleEvidence = document.getElementById("nf-visible-evidence");

    if (visibleEvidence) {
      visibleEvidence.innerHTML = "";

      evidenceForStage(stage).forEach(function (label) {
        var chip = document.createElement("span");
        chip.className = "nf-evidence-chip";
        chip.textContent = label;
        visibleEvidence.appendChild(chip);
      });
    }
  }

  function nearestUsefulContainer(h) {
    if (!h) return null;

    var candidates = [];
    var el = h.parentElement;

    while (
      el &&
      el !== document.body &&
      !el.hasAttribute("data-nf-cockpit") &&
      candidates.length < 5
    ) {
      candidates.push(el);
      el = el.parentElement;
    }

    for (var i = 0; i < candidates.length; i++) {
      var c = candidates[i];
      var rect = c.getBoundingClientRect();

      if (
        rect.width > 240 &&
        rect.height > 80 &&
        rect.height < window.innerHeight * 2.4
      ) {
        return c;
      }
    }

    return h.parentElement;
  }

  function decoratePanels() {
    document
      .querySelectorAll("h1,h2,h3,h4,h5,[role='heading']")
      .forEach(function (h) {
        if (!isVisible(h)) return;

        var t = cleanText(h).toLowerCase();
        if (!t) return;

        var panel = nearestUsefulContainer(h);
        if (!panel || panel.hasAttribute("data-nf-cockpit")) return;

        var kind = null;
        var label = null;

        if (t.includes("review complete")) {
          kind = "nf-panel-complete";
          label = "Session";
        } else if (
          t.includes("final action") ||
          t.includes("final disposition")
        ) {
          kind = "nf-panel-decision";
          label = "Reviewer disposition";
        } else if (
          t.includes("evidence passport") ||
          t.includes("provenance")
        ) {
          kind = "nf-panel-provenance";
          label = "Evidence provenance";
        } else if (
          t.includes("ai recommendation") ||
          t.includes("ai review")
        ) {
          kind = "nf-panel-ai";
          label = "AI-derived evidence";
        } else if (t.includes("initial judgment")) {
          kind = "nf-panel-decision";
          label = "Reviewer judgment";
        } else if (
          t.includes("mri") ||
          t.includes("evidence") ||
          t.includes("longitudinal") ||
          t.includes("case brief")
        ) {
          kind = "nf-panel-evidence";
          label = "Observed evidence";
        }

        if (!kind) return;

        panel.classList.add("nf-auto-panel", kind);

        if (!panel.querySelector(":scope > .nf-panel-label")) {
          var tag = document.createElement("div");
          tag.className = "nf-panel-label";
          tag.setAttribute("aria-hidden", "true");
          tag.textContent = label;
          panel.insertBefore(tag, panel.firstChild);
        }
      });
  }

  function decorateActions() {
    document
      .querySelectorAll("button,input[type='button'],input[type='submit']")
      .forEach(function (b) {
        if (isCockpit(b)) return;

        var t = (
          ((b.value || "") + " " + (b.textContent || ""))
        ).toLowerCase();

        if (
          t.includes("back") ||
          t.includes("previous") ||
          t.includes("download") ||
          t.includes("export")
        ) {
          b.classList.add("nf-secondary-action");
        }
      });
  }

  function refresh() {
    var studyText = visibleStudyText();
    var caseNo = findCurrentCase(studyText);
    var stage = detectStage();

    updateCaseUI(caseNo);
    updateStageUI(stage);
    normalizeSetupPresentation(stage);
    decoratePanels();
    decorateActions();
  }

  var queued = false;

  function scheduleRefresh() {
    if (queued) return;
    queued = true;

    window.requestAnimationFrame(function () {
      queued = false;
      refresh();
    });
  }

  if (document.readyState === "loading") {
    document.addEventListener("DOMContentLoaded", refresh, { once: true });
  } else {
    refresh();
  }

  var observer = new MutationObserver(scheduleRefresh);

  observer.observe(document.body, {
    childList: true,
    subtree: true,
    characterData: true,
    attributes: true,
    attributeFilter: [
      "hidden",
      "style",
      "class",
      "aria-hidden",
      "aria-current",
      "aria-selected"
    ]
  });
})();
</script>
<!-- NF_UI_COCKPIT_SCRIPT_END -->
"""

print("✅ Corrected runtime intelligence defined.")
print("✅ Current stage comes from active content, not future step labels.")
print("✅ Evidence chips are stage-gated and cannot reveal future AI/provenance.")

✅ Corrected runtime intelligence defined.
✅ Current stage comes from active content, not future step labels.
✅ Evidence chips are stage-gated and cannot reveal future AI/provenance.


In [8]:
# Cell 8 — Inject shell into CLEAN frozen HTML

def inject_before_close(html: str, tag: str, block: str) -> str:
    ms = list(re.finditer(rf"</{tag}\s*>", html, flags=re.I))
    assert ms, f"Closing </{tag}> not found."
    m = ms[-1]
    return html[:m.start()] + "\n" + block.strip() + "\n" + html[m.start():]

def inject_after_body_open(html: str, block: str) -> str:
    m = re.search(r"<body\b[^>]*>", html, flags=re.I)
    assert m, "<body> not found."
    return html[:m.end()] + "\n" + block.strip() + "\n" + html[m.end():]

candidate_html = source_html

# Style in head.
candidate_html = inject_before_close(candidate_html, "head", COCKPIT_STYLE)

# Application shell: purely read-only.
candidate_html = inject_after_body_open(
    candidate_html,
    LEFT_RAIL + "\n" + TOP_BAR + "\n" + RIGHT_RAIL
)

# UI intelligence after the original application markup/scripts.
candidate_html = inject_before_close(candidate_html, "body", COCKPIT_SCRIPT)

assert "NF_UI_COCKPIT_STYLE_BEGIN" in candidate_html
assert "NF_UI_COCKPIT_LEFT_BEGIN" in candidate_html
assert "NF_UI_COCKPIT_RIGHT_BEGIN" in candidate_html
assert "NF_UI_COCKPIT_SCRIPT_BEGIN" in candidate_html

print("✅ Evidence Cockpit candidate HTML assembled.")
print("Candidate bytes:", len(candidate_html.encode("utf-8")))

✅ Evidence Cockpit candidate HTML assembled.
Candidate bytes: 45969


In [9]:
# Cell 9 — Write candidate app with all original assets untouched
if CANDIDATE_APP.exists():
    shutil.rmtree(CANDIDATE_APP)

shutil.copytree(SOURCE_APP, CANDIDATE_APP)
CANDIDATE_INDEX = CANDIDATE_APP / "index.html"
CANDIDATE_INDEX.write_text(candidate_html, encoding="utf-8")

print("✅ Candidate app:", CANDIDATE_APP)

✅ Candidate app: /content/drive/MyDrive/neurofhir-qc/wish_extension/final_wish_pilot/participant_app_evidence_cockpit_candidate


In [10]:
# Cell 10 — HARD protocol-integrity gate

# 1. Existing application script blocks must be identical and in the same order.
scripts_after = SCRIPT_RE.findall(candidate_html)
original_scripts_after = [
    s for s in scripts_after
    if "nf-evidence-cockpit-script" not in s
]
script_hashes_after = [sha256_text(s) for s in original_scripts_after]

assert script_hashes_after == script_hashes_before, (
    "❌ An existing application script changed. STOP."
)

# 2. Original controls must remain identical.
soup_after = BeautifulSoup(candidate_html, "html.parser")

# Cockpit intentionally contains no input/select/textarea/button controls.
cockpit_controls = []
for shell in soup_after.select("[data-nf-cockpit]"):
    cockpit_controls += shell.find_all(["input", "select", "textarea", "button"])

assert len(cockpit_controls) == 0, (
    "Cockpit shell unexpectedly contains participant controls."
)

controls_after = control_signature(soup_after)
assert controls_after == controls_before, (
    "❌ Participant control signature changed. STOP."
)

# 3. Critical study tokens that existed before still exist.
for token, existed in tokens_present_before.items():
    if existed:
        assert token.lower() in candidate_html.lower(), (
            f"Critical token disappeared: {token}"
        )

# 4. All non-index assets remain byte-identical.
asset_failures = []
for src in sorted(SOURCE_APP.rglob("*")):
    if not src.is_file():
        continue
    rel = src.relative_to(SOURCE_APP)
    if rel.as_posix() == "index.html":
        continue
    dst = CANDIDATE_APP / rel
    if not dst.exists():
        asset_failures.append(f"missing:{rel}")
    elif sha256_file(src) != sha256_file(dst):
        asset_failures.append(f"changed:{rel}")

assert not asset_failures, "\n".join(asset_failures[:50])

print("✅ Existing application scripts unchanged")
print("✅ Participant controls unchanged")
print("✅ Critical study fields preserved")
print("✅ Non-index assets unchanged")
print("✅ HARD PROTOCOL-INTEGRITY GATE: PASS")

✅ Existing application scripts unchanged
✅ Participant controls unchanged
✅ Critical study fields preserved
✅ Non-index assets unchanged
✅ HARD PROTOCOL-INTEGRITY GATE: PASS


In [11]:
# Cell 11 — Leakage / priming / network-dependency gate

FORBIDDEN_TEXT = [
    "source_case_id",
    "ai_correctness",
    "reference_workflow_disposition_path_b",
    "path_a_reference_status",
    "final_researcher_scenario_key",
    "researcher_only",
    "reference_volume",
]

FORBIDDEN_FILENAMES = [
    "researcher",
    "answer_key",
    "scenario_key",
    "reference_workflow",
]

TEXT_SUFFIXES = {
    ".html", ".htm", ".js", ".mjs", ".cjs", ".json",
    ".css", ".txt", ".csv", ".md", ".xml"
}

leaks = []

for p in CANDIDATE_APP.rglob("*"):
    if not p.is_file():
        continue
    rel = p.relative_to(CANDIDATE_APP).as_posix().lower()

    for marker in FORBIDDEN_FILENAMES:
        if marker in rel:
            leaks.append(f"filename:{rel} -> {marker}")

    if p.suffix.lower() in TEXT_SUFFIXES:
        txt = p.read_text(encoding="utf-8", errors="ignore").lower()
        for marker in FORBIDDEN_TEXT:
            if marker in txt:
                leaks.append(f"content:{rel} -> {marker}")

assert not leaks, (
    "❌ Researcher-only leakage detected:\n" + "\n".join(leaks[:50])
)

# Fixed shell must not reveal the randomized sequence.
fixed_shell = (LEFT_RAIL + TOP_BAR + RIGHT_RAIL).lower()
for marker in [
    "evidence-first",
    "ai-first",
    "sequence a",
    "sequence b",
    "correct answer",
    "expected disposition",
]:
    assert marker not in fixed_shell, f"Priming/condition marker in shell: {marker}"

# No external design libraries, analytics, or fonts.
lower = candidate_html.lower()
for bad in [
    "fonts.googleapis.com",
    "fonts.gstatic.com",
    "fontawesome",
    "cdnjs.cloudflare.com",
    "unpkg.com",
    "google-analytics.com",
    "googletagmanager.com",
]:
    assert bad not in lower, f"Unexpected external UI/analytics dependency: {bad}"

print("✅ Researcher-only leakage: NONE")
print("✅ Hidden sequence disclosure: NONE")
print("✅ External UI/analytics dependency added: NONE")
print("✅ Leakage / priming gate: PASS")

✅ Researcher-only leakage: NONE
✅ Hidden sequence disclosure: NONE
✅ External UI/analytics dependency added: NONE
✅ Leakage / priming gate: PASS


In [12]:
# Cell 12 — Static structural QA
required_ids = [
    "nf-cockpit-left",
    "nf-cockpit-top",
    "nf-cockpit-right",
    "nf-case-grid",
    "nf-visible-evidence",
    "nf-progress-fill",
]

for rid in required_ids:
    assert f'id="{rid}"' in candidate_html, f"Missing cockpit element: {rid}"

assert candidate_html.count('class="nf-case-dot"') == 12

# Research disclaimer.
assert "not for patient care or diagnostic use" in candidate_html.lower()

print("✅ Left case rail present")
print("✅ Top context bar present")
print("✅ Right review console present")
print("✅ 12-case visual navigator present")
print("✅ Responsive CSS present")
print("✅ Research disclaimer present")
print("✅ Static structural QA: PASS")

✅ Left case rail present
✅ Top context bar present
✅ Right review console present
✅ 12-case visual navigator present
✅ Responsive CSS present
✅ Research disclaimer present
✅ Static structural QA: PASS


In [13]:
# Cell 13 — Promote Evidence Cockpit to frozen participant app
if PROMOTE_TO_FROZEN_APP:
    # Promote only index.html. Original assets were already checked byte-for-byte.
    shutil.copy2(CANDIDATE_INDEX, CURRENT_INDEX)

    promoted_html = CURRENT_INDEX.read_text(encoding="utf-8")
    assert "NF_UI_COCKPIT_STYLE_BEGIN" in promoted_html
    assert "NF_UI_FACELIFT_STYLE_BEGIN" not in promoted_html, (
        "15A1 and 15A2 appear to be stacked. STOP."
    )

    print("✅ Evidence Cockpit promoted to frozen participant app.")
    print("Frozen index:", CURRENT_INDEX)
else:
    print("ℹ️ PROMOTE_TO_FROZEN_APP=False")
    print("Candidate retained at:", CANDIDATE_APP)

✅ Evidence Cockpit promoted to frozen participant app.
Frozen index: /content/drive/MyDrive/neurofhir-qc/wish_extension/final_wish_pilot/participant_app/index.html


In [14]:
# Cell 14 — Interface freeze audit
audit = {
    "artifact": "NeuroFHIR-Review Evidence Cockpit UI",
    "ui_version": "15A2-evidence-cockpit-v1",
    "generated_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(),
    "source_clean_app": str(SOURCE_APP),
    "candidate_app": str(CANDIDATE_APP),
    "promoted_frozen_app": str(FROZEN_APP) if PROMOTE_TO_FROZEN_APP else None,
    "pre_ui_index_sha256": source_hash,
    "post_ui_index_sha256": sha256_file(
        CURRENT_INDEX if PROMOTE_TO_FROZEN_APP else CANDIDATE_INDEX
    ),
    "existing_application_script_hashes_preserved": True,
    "participant_control_signature_preserved": True,
    "non_index_assets_preserved": True,
    "researcher_only_leakage": False,
    "hidden_sequence_disclosed": False,
    "external_ui_dependencies_added": False,
    "ui_components": {
        "left_case_rail": True,
        "top_context_bar": True,
        "right_review_console": True,
        "center_evidence_workspace": True,
        "case_progress_grid": 12,
        "responsive_layout": True,
        "ai_visual_privilege": False,
    },
    "real_participant_first_id": "P003",
}

audit_path = (
    WISH_ROOT / "final_wish_pilot" / "ui_evidence_cockpit_audit.json"
)
audit_path.write_text(json.dumps(audit, indent=2), encoding="utf-8")

print("✅ UI audit:", audit_path)

✅ UI audit: /content/drive/MyDrive/neurofhir-qc/wish_extension/final_wish_pilot/ui_evidence_cockpit_audit.json


## After 15A2 passes

Do these in this order:

1. **Rerun Notebook 15A** so GitHub Pages receives the new frozen `index.html`.
2. Open `https://sanghati23.github.io/neurofhir-qc/wish-review/` yourself on a laptop.
3. Check the interface at normal browser width and a narrow/mobile width.
4. Run corrected **Notebook 15B**.
5. Do **not** send the link to Teri until the live version looks right and QA passes.

Because P003 has not started, we are still free to polish presentation. Once real participant collection begins, this UI version becomes part of the frozen protocol.

In [15]:
# Cell 15 — Final gate
final_ok = (
    script_hashes_after == script_hashes_before
    and controls_after == controls_before
    and not asset_failures
    and not leaks
)

if PROMOTE_TO_FROZEN_APP:
    final_ok = (
        final_ok
        and "NF_UI_COCKPIT_STYLE_BEGIN"
            in CURRENT_INDEX.read_text(encoding="utf-8")
        and "NF_UI_FACELIFT_STYLE_BEGIN"
            not in CURRENT_INDEX.read_text(encoding="utf-8")
    )

print("=" * 80)
print("NOTEBOOK 15A2 — NEUROFHIR-REVIEW EVIDENCE COCKPIT")
print("=" * 80)
print("Clean pre-UI source:                         PASS")
print("Left case-navigation rail:                   PASS")
print("Top contextual review bar:                   PASS")
print("Right read-only review console:              PASS")
print("Center evidence workspace styling:           PASS")
print("12-case visual progress:                     PASS")
print("Responsive desktop/tablet/mobile rules:      PASS")
print("Existing application scripts unchanged:      PASS")
print("Participant controls unchanged:              PASS")
print("Non-index assets unchanged:                  PASS")
print("Researcher-only leakage:                     PASS")
print("Hidden condition disclosure:                 PASS")
print("AI visual privilege introduced:              NO")
print("Promoted to frozen participant app:         ",
      "YES" if PROMOTE_TO_FROZEN_APP else "NO")
print("=" * 80)

assert final_ok

print("✅ NOTEBOOK 15A2 EVIDENCE COCKPIT GATE: TRUE")
print()
print("NEXT → Notebook 15A redeploy → live visual inspection → Notebook 15B QA")

NOTEBOOK 15A2 — NEUROFHIR-REVIEW EVIDENCE COCKPIT
Clean pre-UI source:                         PASS
Left case-navigation rail:                   PASS
Top contextual review bar:                   PASS
Right read-only review console:              PASS
Center evidence workspace styling:           PASS
12-case visual progress:                     PASS
Responsive desktop/tablet/mobile rules:      PASS
Existing application scripts unchanged:      PASS
Participant controls unchanged:              PASS
Non-index assets unchanged:                  PASS
Researcher-only leakage:                     PASS
Hidden condition disclosure:                 PASS
AI visual privilege introduced:              NO
Promoted to frozen participant app:          YES
✅ NOTEBOOK 15A2 EVIDENCE COCKPIT GATE: TRUE

NEXT → Notebook 15A redeploy → live visual inspection → Notebook 15B QA
